In [12]:
import os
import shutil
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from datasets import load_dataset
from tqdm.auto import tqdm

In [ ]:
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False


if IS_COLAB:
    BASE_DIR = Path("/content")
else:
    def _find_project_root():
        for candidate in (Path.cwd(), *Path.cwd().parents):
            if (candidate / "configs" / "config.yaml").exists():
                return candidate
        return Path.cwd()
    BASE_DIR = _find_project_root()

DATA_ROOT = BASE_DIR / "data"
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "validation"
CHECKPOINT_PATH = BASE_DIR / "results" / "checkpoints" / "VGG_16.pth"

_yaml_path = BASE_DIR / "configs" / "config.yaml"
if _yaml_path.exists():
    import yaml
    with open(_yaml_path, "r", encoding="utf-8") as _f:
        config = yaml.safe_load(_f)
else:
    config = {
        "training": {
            "epochs": 40,
            "batch_size": 64,
            "learning_rate": 0.01,
            "momentum": 0.9,
            "weight_decay": 5e-4,
        },
        "testing": {"batch_size": 64},
        "train_dataloader": {"num_workers": 8},
        "test_dataloader": {"num_workers": 8},
    }

In [ ]:
import torch
from torch import nn


class VGG_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.convblocks = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(25088, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 100),
        )

    def forward(self, x):
        return self.classifier(self.convblocks(x))


In [ ]:
def _split_has_images(split_dir):
    if not split_dir.is_dir():
        return False
    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue
        if any(p.is_file() for p in class_dir.iterdir()):
            return True
    return False


def _dataset_is_ready():
    return _split_has_images(TRAIN_DIR) and _split_has_images(VAL_DIR)


def download_datasets():
    if _dataset_is_ready():
        return

    if TRAIN_DIR.exists():
        shutil.rmtree(TRAIN_DIR)
    if VAL_DIR.exists():
        shutil.rmtree(VAL_DIR)

    print("ImageNet-100 not found locally. Downloading from Hugging Face Hub...")
    dataset = load_dataset("clane9/imagenet-100")
    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    VAL_DIR.mkdir(parents=True, exist_ok=True)

    for split, target_dir in (("train", TRAIN_DIR), ("validation", VAL_DIR)):
        for example in tqdm(dataset[split], desc=f"Saving {split}"):
            label = str(example["label"])
            image = example["image"]
            class_dir = target_dir / label
            class_dir.mkdir(parents=True, exist_ok=True)
            image_path = class_dir / f"{hash(image.tobytes())}.jpg"
            image.save(image_path)


def get_datasets():
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose(
        [
            transforms.Resize(256),
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
        ]
    )
    test_transform = transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
        ]
    )
    download_datasets()
    train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
    test_dataset = ImageFolder(VAL_DIR, transform=test_transform)
    return train_dataset, test_dataset


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def use_pin_memory(device, workers):
    return device.type == "cuda" and not (os.name == "nt" and workers > 0)


def make_data_loader(dataset, batch_size, workers, shuffle, device):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=use_pin_memory(device, workers),
        persistent_workers=workers > 0,
    )

In [ ]:
training_config = config["training"]
train_dataloader_config = config["train_dataloader"]


def train():
    torch.set_float32_matmul_precision("high")

    device = get_device()
    print(f"Using device: {device}")
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

    train_dataset, _ = get_datasets()
    train_loader = make_data_loader(
        train_dataset,
        batch_size=training_config["batch_size"],
        workers=train_dataloader_config["num_workers"],
        shuffle=True,
        device=device,
    )

    model = VGG_16().to(device)
    if hasattr(torch, "compile") and device.type == "cuda":
        model = torch.compile(model)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=training_config["learning_rate"],
        momentum=training_config["momentum"],
        weight_decay=training_config["weight_decay"],
    )

    for epoch in tqdm(range(training_config["epochs"]), desc="Training epochs"):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for inputs, targets in train_loader:
            non_blocking = use_pin_memory(device, train_dataloader_config["num_workers"])
            inputs = inputs.to(device, non_blocking=non_blocking)
            targets = targets.to(device, non_blocking=non_blocking)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            predictions = torch.argmax(outputs, dim=1)
            train_correct += (predictions == targets).sum().item()
            train_total += targets.size(0)

        epoch_loss = train_loss / train_total
        epoch_accuracy = train_correct / train_total
        print(
            f"Epoch [{epoch + 1}/{training_config['epochs']}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}"
        )

        model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model
        torch.save(model_to_save.state_dict(), CHECKPOINT_PATH)

In [17]:
testing_config = config["testing"]
test_dataloader_config = config["test_dataloader"]


def test():
    device = get_device()
    _, test_dataset = get_datasets()

    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found at {CHECKPOINT_PATH}. Run train() first to generate it."
        )

    test_loader = make_data_loader(
        test_dataset,
        batch_size=testing_config["batch_size"],
        workers=test_dataloader_config["num_workers"],
        shuffle=False,
        device=device,
    )

    model = VGG_16().to(device)
    model.load_state_dict(
        torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    )
    model.eval()

    correct = 0
    total = 0
    non_blocking = use_pin_memory(device, test_dataloader_config["num_workers"])

    with torch.inference_mode():
        for inputs, targets in tqdm(test_loader, desc="Testing"):
            inputs  = inputs.to(device, non_blocking=non_blocking)
            targets = targets.to(device, non_blocking=non_blocking)
            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == targets).sum().item()
            total   += targets.size(0)

    test_accuracy = correct / total
    print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
import random
import gradio as gr
from PIL import Image as PILImage


_infer_model = None
_infer_device = None
_val_dataset = None


def _get_infer_resources():
    global _infer_model, _infer_device, _val_dataset
    if _infer_model is None:
        if not CHECKPOINT_PATH.exists():
            raise FileNotFoundError(
                f"Checkpoint not found at {CHECKPOINT_PATH}. Run train() first."
            )
        _infer_device = get_device()
        _infer_model = VGG_16().to(_infer_device)
        _infer_model.load_state_dict(
            torch.load(CHECKPOINT_PATH, map_location=_infer_device, weights_only=True)
        )
        _infer_model.eval()
        _, _val_dataset = get_datasets()
    return _infer_model, _infer_device, _val_dataset


imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

_crop_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
_infer_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)


def predict_random_five():
    model, device, val_dataset = _get_infer_resources()
    class_names = val_dataset.classes
    indices = random.sample(range(len(val_dataset)), 5)
    gallery_items = []

    for idx in indices:
        img_path, true_label = val_dataset.samples[idx]
        pil_img = PILImage.open(img_path).convert("RGB")

        tensor = _infer_transform(pil_img).unsqueeze(0).to(device)
        with torch.inference_mode():
            probs = torch.softmax(model(tensor), dim=1)
            top_prob, top_idx = probs[0].max(0)

        pred_class = class_names[top_idx.item()]
        true_class = class_names[true_label]
        caption = (
            f"Predicted: {pred_class}  ({top_prob.item() * 100:.1f}%)\n"
            f"True label: {true_class}"
        )
        gallery_items.append((_crop_transform(pil_img), caption))

    return gallery_items


with gr.Blocks(title="VGG-16 · ImageNet-100 Predictor") as demo:
    gr.Markdown(
        "## VGG-16 · Random 5-Image Predictor\n"
        "Click **Sample & Predict** to draw 5 random validation images, run VGG-16\n"
        "inference, and display the **predicted class** with its **confidence probability**."
    )
    sample_btn = gr.Button("Sample & Predict", variant="primary")
    gallery = gr.Gallery(
        label="Predictions",
        columns=5,
        rows=1,
        height="auto",
        object_fit="contain",
        show_label=True,
    )
    sample_btn.click(fn=predict_random_five, outputs=gallery)

demo.launch(share=IS_COLAB)